# Eksplorasi YOLO26 pada Dataset v2 (UR Fall + MCF, buatan Laila)

Notebook ini menjalankan eksplorasi YOLO26 secara cepat di Kaggle. Isinya tiga bagian yang bisa dinyalakan satu per satu:

| Bagian | Isi | Perlu training? | Perkiraan waktu (T4) |
|---|---|---|---|
| **Y1** | Fine-tune `yolo26n` (atau s) di dataset v2: **3 kelas** vs **2 kelas**, lalu evaluasi lying per sumber dan per kamera | Ya | ±40 menit per run |
| **Y0 pose** | YOLO26-pose (n, s, m) tanpa training pada frame lying, di beberapa resolusi | Tidak | ±5 menit |
| **Y0 person** | Detektor orang (YOLO11 vs YOLO26) pada frame lying, untuk menilai pembuat kotak otomatis | Tidak | ±5 menit |

**Alur kerja: LOKAL dulu, Kaggle hanya bila lokal terlalu lama.**
- Notebook ini berjalan di kedua tempat (otomatis mengenali lingkungan). Lokal: buka di VS Code dengan kernel `.venv` proyek; dataset dibaca dari `Data Baru/` di root repo dan keluaran disimpan di `docs/internal/out/yolo26/` (tidak ikut git).
- **Aturan pindah ke Kaggle:** jalankan lokal, lihat waktu epoch ke-1 dan ke-2 di progress bar. Jika `waktu epoch x 70` lebih dari sekitar 2 jam, hentikan dan jalankan di Kaggle (T4) untuk run tersebut. Y0 dan evaluasi umumnya cukup cepat di lokal.

**Cara pakai (Kaggle, bila diperlukan)**
1. Zip isi folder `Data Baru` dengan Python (`shutil.make_archive`, jangan `Compress-Archive`) dan upload sebagai Kaggle Dataset (private).
2. Pada notebook: *Add Data* untuk dataset itu, **Accelerator: GPU T4**, dan **Internet: On** (untuk mengunduh bobot YOLO26 dan, bila perlu, memperbarui library).
3. Jalankan dulu dengan `SMOKE = True` (±3 menit) untuk memastikan semuanya berjalan. Lalu ubah `SMOKE = False` dan **Run All**.
4. Unduh `y26_outputs.zip` dari tab Output dan kirim ke Raja.

**Jaga keadilan perbandingan dengan YOLO11 (Laila):** dataset yang sama (cek sidik jari di sel 3), resep latih yang sama (sel 2), dan metrik evaluasi yang sama (sel 5b = salinan `eval_lying.py` milik Laila; sel 5 adalah diagnostik tambahan per kamera). Satu run saja tidak cukup untuk menyimpulkan, karena test kita kecil (lihat catatan di sel 5).

In [ ]:
# ---- 1. Setup & pemeriksaan versi -------------------------------------
import importlib, subprocess, sys, os, re, json, math, time, csv, zipfile, shutil, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle").exists()

def repo_root():
    here = Path.cwd().resolve()
    for q in [here, *here.parents]:
        if (q / ".git").exists():
            return q
    return here

def _pip_upgrade():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])

import ultralytics
from ultralytics import YOLO
import torch
print("Lingkungan:", "KAGGLE" if ON_KAGGLE else "LOKAL", "| repo:", repo_root())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (aktifkan GPU!)")

try:
    YOLO("yolo26n.pt")
    print("YOLO26 tersedia.")
except Exception as e:
    print("YOLO26 belum bisa dimuat:", str(e)[:200])
    print("Memperbarui ultralytics ... (setelah ini: Restart session lalu Run All)")
    _pip_upgrade()
    raise SystemExit("Restart session dan jalankan ulang dari atas.")

In [ ]:
# ---- 2. KONFIGURASI (ubah di sini saja) --------------------------------
SMOKE = True            # True = uji cepat (data 5%, 2 epoch). Ubah ke False untuk run sebenarnya.

# Daftar run Y1: (nama_run, varian_label, bobot, seed [, {override resep}])
#   varian: "3cls"       = normal / transitional / lying (seperti dataset aslinya)
#           "2cls_merge" = bukan-lying / lying (transitional digabung ke bukan-lying)
#           "2cls_drop"  = normal vs lying, frame transitional DIBUANG (seperti mode 2 kelas Laila)
RUNS = [
    ("y26n_3cls_s0",   "3cls",       "yolo26n.pt", 0),
    ("y26n_2cls_s0",   "2cls_merge", "yolo26n.pt", 0),
    # ("y26n_2cls_drop_s0", "2cls_drop", "yolo26n.pt", 0),
    # ("y26s_3cls_s0",   "3cls",       "yolo26s.pt", 0),
    # ("y26n_3cls_s1",   "3cls",       "yolo26n.pt", 1),    # ulangi seed untuk mengukur noise
    # ("y26n_3cls_auto_s0", "3cls",     "yolo26n.pt", 0, {"optimizer": "auto"}),  # resep bawaan YOLO26 (adil bagi YOLO26)
]

# Resep sama dengan EXP-006 (juara bounding box YOLO11), supaya adil terhadap YOLO11 yang dijalankan Laila.
# Catatan: optimizer DIKUNCI ke AdamW (bukan "auto") agar YOLO26 dan YOLO11 sebanding.
RECIPE = dict(epochs=70, patience=30, batch=16, imgsz=640, optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True,
              cls=1.0, hsv_v=0.6, hsv_s=0.7, hsv_h=0.015, bgr=0.2, erasing=0.4,
              flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10, workers=2 if ON_KAGGLE else 0)   # Windows/lokal: 0 agar aman

RUN_Y1 = True
RUN_Y0_POSE = False      # YOLO26-pose tanpa training di frame lying
RUN_Y0_PERSON = False    # detektor orang di frame lying (menilai alat pembuat kotak dataset)

EVAL_CONF = 0.25
Y0_WEIGHTS_POSE = ["yolo11n-pose.pt", "yolo26n-pose.pt", "yolo26s-pose.pt", "yolo26m-pose.pt"]
Y0_WEIGHTS_PERSON = ["yolo11n.pt", "yolo11m.pt", "yolo26n.pt", "yolo26s.pt", "yolo26m.pt"]
Y0_IMGSZ = [640, 960, 1280]

# Sidik jari dataset yang BENAR (dihitung dari folder Data Baru lokal Raja, selaras dengan manifest Laila terbaru)
EXPECTED = {"n_images": 8645, "files_sha256": "6a400a62942aa767", "labels_sha256": "6ff25b047052815c", "counts": {"test/0": 1677, "test/1": 1460, "test/2": 264, "train/0": 1802, "train/1": 1164, "train/2": 802, "val/0": 578, "val/1": 770, "val/2": 128}}

if SMOKE:
    RECIPE = {**RECIPE, "epochs": 2, "patience": 2, "fraction": 0.05}
    RUNS = RUNS[:2]
print("SMOKE:", SMOKE, "| RUNS:", [r[0] for r in RUNS])

In [ ]:
# ---- 3. Cari dataset & verifikasi bahwa ini data terbaru Laila ----------
import hashlib
from collections import Counter

def find_dataset_root():
    bases = [Path("/kaggle/input"), repo_root() / "Data Baru", Path("Data Baru"), Path(".")]
    for b in bases:
        if not b.exists():
            continue
        for cand in [b, *b.rglob("*")]:
            if cand.is_dir() and (cand / "images" / "test").is_dir() and (cand / "labels" / "test").is_dir():
                return cand
    raise FileNotFoundError("Folder dataset (images/ dan labels/) tidak ditemukan. Cek 'Add Data'.")

ROOT = find_dataset_root()
names, lab, cnt = [], hashlib.sha256(), Counter()
for sp in ("train", "val", "test"):
    names += [f"{sp}/{p.name}" for p in sorted((ROOT / "images" / sp).glob("*.jpg"))]
    for l in sorted((ROOT / "labels" / sp).glob("*.txt")):
        t = l.read_text().strip()
        lab.update(f"{sp}/{l.name}:{t}\n".encode())
        cnt[f"{sp}/{t.split()[0]}"] += 1
got = {"n_images": len(names),
       "files_sha256": hashlib.sha256("\n".join(names).encode()).hexdigest()[:16],
       "labels_sha256": lab.hexdigest()[:16],
       "counts": dict(sorted(cnt.items()))}
print("Dataset:", ROOT)
print(json.dumps(got, indent=1))
if EXPECTED:
    ok = all(got.get(k) == EXPECTED.get(k) for k in ("n_images", "files_sha256", "labels_sha256", "counts"))
    print("\nSIDIK JARI COCOK DENGAN DATA TERBARU LAILA." if ok else
          "\n!!! SIDIK JARI BERBEDA: dataset ini BUKAN versi yang disepakati. Hentikan dan konfirmasi ke Raja/Laila.")
    assert ok, "Dataset tidak cocok dengan versi yang disepakati"

In [ ]:
# ---- 4. Membangun varian dataset (label 3 kelas / 2 kelas) tanpa menggandakan gambar ----
WORK = Path("/kaggle/working") if ON_KAGGLE else repo_root() / "docs" / "internal" / "out" / "yolo26"
WORK = WORK.resolve()
WORK.mkdir(parents=True, exist_ok=True)
DS = WORK / "ds"
OUTD = WORK / "out"
OUTD.mkdir(exist_ok=True)


def _link(src, dst):
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def build_variant(variant):
    """Membuat DS/<variant>/images/<split> (tautan) dan labels/<split> (ditulis ulang). Mengembalikan path data.yaml."""
    out = DS / variant
    if (out / "data.yaml").exists():
        return out / "data.yaml"
    for sp in ("train", "val", "test"):
        (out / "labels" / sp).mkdir(parents=True, exist_ok=True)
        (out / "images" / sp).mkdir(parents=True, exist_ok=True)
        for lbl in (ROOT / "labels" / sp).glob("*.txt"):
            parts = lbl.read_text().split()
            c = int(parts[0])
            if variant == "3cls":
                nc = c
            elif variant == "2cls_merge":
                nc = 1 if c == 2 else 0
            elif variant == "2cls_drop":
                if c == 1:
                    continue
                nc = 1 if c == 2 else 0
            else:
                raise ValueError(variant)
            (out / "labels" / sp / lbl.name).write_text(" ".join([str(nc)] + parts[1:]) + "\n")
            img = ROOT / "images" / sp / (lbl.stem + ".jpg")
            _link(img, out / "images" / sp / img.name)
    names = ({0: "normal", 1: "transitional", 2: "lying_on_ground"} if variant == "3cls"
             else {0: "not_lying", 1: "lying_on_ground"} if variant == "2cls_merge"
             else {0: "normal", 1: "lying_on_ground"})
    yaml = f"path: {out}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\nnames:\n"
    yaml += "".join(f"  {k}: {v}\n" for k, v in names.items())
    (out / "data.yaml").write_text(yaml)
    n = {sp: len(list((out / "images" / sp).glob("*.jpg"))) for sp in ("train", "val", "test")}
    print(f"[{variant}] gambar:", n)
    return out / "data.yaml"

In [ ]:
# ---- 5. Evaluasi seragam (SAMA untuk YOLO26 dan YOLO11) -----------------
# Selalu memakai test 3 kelas ASLI sebagai kebenaran; yang diukur: apakah gambar lying terdeteksi sebagai lying.
# Catatan keandalan: test lying MCF berasal dari hanya 4 skenario jatuh (8 kamera masing-masing);
# jumlah frame menggelembungkan rasa percaya. Bacalah per kamera dan ulangi dengan beberapa seed.
def src_of(n):
    return "mcf" if n.startswith("mcf_") else "urfall"

def cam_of(n):
    m = re.match(r"mcf_s\d+_(c\d+)_", n)
    return m.group(1) if m else "cam0"

def eval_diag(weights, variant, conf=EVAL_CONF, imgsz=640):
    lying_cls = 2 if variant == "3cls" else 1
    model = YOLO(str(weights))
    imgs = sorted((ROOT / "images" / "test").glob("*.jpg"))
    rows = []
    t0 = time.time()
    for i in range(0, len(imgs), 32):
        ch = imgs[i:i + 32]
        for p, r in zip(ch, model([str(x) for x in ch], conf=conf, imgsz=imgsz, verbose=False)):
            gt = int((ROOT / "labels" / "test" / (p.stem + ".txt")).read_text().split()[0])
            if len(r.boxes):
                cls = r.boxes.cls.cpu().numpy().astype(int)
                cf = r.boxes.conf.cpu().numpy()
                top = int(cls[int(cf.argmax())])
                rows.append(dict(img=p.name, gt=gt, none=0, top_lying=int(top == lying_cls),
                                 any_lying=int((cls == lying_cls).any()), top=top,
                                 lying_conf=float(cf[cls == lying_cls].max()) if (cls == lying_cls).any() else 0.0))
            else:
                rows.append(dict(img=p.name, gt=gt, none=1, top_lying=0, any_lying=0, top=-1, lying_conf=0.0))
    ms = (time.time() - t0) / len(imgs) * 1000
    df = pd.DataFrame(rows)
    df["source"] = df.img.map(src_of)
    df["camera"] = df.img.map(cam_of)
    return df, ms

def summarize(df, tag):
    def one(d, name):
        y = (d["gt"] == 2).values
        out = {"kelompok": name, "n": len(d), "lying_n": int(y.sum())}
        for col in ("top_lying", "any_lying"):
            p = d[col].values.astype(bool)
            tp, fp = int((p & y).sum()), int((p & ~y).sum())
            out[col + "_R"] = round(tp / max(1, y.sum()), 3)
            out[col + "_P"] = round(tp / max(1, tp + fp), 3)
            out[col + "_FP"] = fp
        out["lying_tak_terdeteksi"] = int(d[(d["gt"] == 2) & (d.none == 1)].shape[0])
        return out
    rows = [one(df, "SEMUA")]
    for s, g in df.groupby("source"):
        rows.append(one(g, s))
    for c, g in df[df.source == "mcf"].groupby("camera"):
        rows.append(one(g, "mcf_" + c))
    res = pd.DataFrame(rows)
    res.insert(0, "run", tag)
    return res

In [ ]:
# ---- 5b. Evaluasi BERSAMA dengan Laila: salinan ml/evaluation/eval_lying.py (commit 0bd3762) ----
# Dipakai untuk YOLO26 dan YOLO11 agar angkanya sebanding: lying vs normal, interval kepercayaan bootstrap PER VIDEO/SKENARIO,
# transitional hanya diagnostik. Jika Laila memperbarui skripnya, ganti isi sel ini dengan versi terbaru.
EVAL_LYING_CODE = r'''#!/usr/bin/env python3
"""
eval_lying.py  ->  ml/evaluation/eval_lying.py
===========================================
Evaluasi "lying vs bukan-lying" yang SAMA untuk model 2 kelas dan 3 kelas,
supaya uji A/B bisa diputuskan dengan satu aturan.

Tiap gambar test punya satu orang/satu label. Skor gambar = confidence tertinggi
dari kotak berkelas lying_on_ground (0 bila tak ada). Kelas model dicari lewat
NAMA ('lying_on_ground'), jadi model 2 kelas (id 1) dan 3 kelas (id 2) sama-sama jalan.

Ground truth selalu dari dataset 3 kelas (0 normal, 1 transitional, 2 lying):
  - tabel UTAMA  : lying vs normal saja (frame bersih, tanpa transitional)
  - tabel DIAGNOSTIK: berapa % frame transitional yang diprediksi lying
    (transitional ambigu, jadi hanya diagnostik, BUKAN dihitung sebagai false positive utama)

Interval kepercayaan: bootstrap per VIDEO/SKENARIO (bukan per frame), karena frame
dari satu kejadian saling berkorelasi; sampel efektif = jumlah kejadian.

Contoh (di Kaggle, setelah training):
  python eval_lying.py --weights runs/exp_2cls/weights/best.pt \
      --gt-dir /kaggle/input/<dataset-3kelas> --out eval_2cls.json
"""
from __future__ import annotations

import argparse
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

THRESHOLDS = (0.10, 0.25, 0.40, 0.50, 0.60, 0.75)
GT_NORMAL, GT_TRANS, GT_LYING = 0, 1, 2
LYING_NAME = "lying_on_ground"


def video_of(stem: str) -> str:
    m = re.match(r"mcf_s(\d+)_c\d_f\d+$", stem)
    if m:
        return f"mcf-s{m.group(1)}"
    m = re.match(r"urfall_((?:fall|adl)-\d+)_f\d+$", stem)
    if m:
        return f"urfall-{m.group(1)}"
    return stem  # tak dikenal: tiap frame jadi kelompok sendiri


def average_precision(pos: np.ndarray, neg: np.ndarray) -> float:
    """AP lying-vs-normal; skor sama (mis. banyak 0) dihitung sebagai satu ambang."""
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    ap, prev_recall = 0.0, 0.0
    for t in np.unique(np.concatenate([pos, neg]))[::-1]:
        tp, fp = int((pos >= t).sum()), int((neg >= t).sum())
        recall = tp / len(pos)
        ap += (recall - prev_recall) * tp / (tp + fp)
        prev_recall = recall
    return float(ap)


def metrics(scores: np.ndarray, gt: np.ndarray, thr: float) -> dict:
    s_l, s_n, s_t = scores[gt == GT_LYING], scores[gt == GT_NORMAL], scores[gt == GT_TRANS]
    tp, fp = int((s_l >= thr).sum()), int((s_n >= thr).sum())
    return {
        "recall": tp / len(s_l) if len(s_l) else float("nan"),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "fp_normal": fp,
        "fpr_normal": fp / len(s_n) if len(s_n) else float("nan"),
        "trans_as_lying": float((s_t >= thr).mean()) if len(s_t) else float("nan"),
        "ap": average_precision(s_l, s_n),
    }


def evaluate(scores, gt, videos, n_boot=500, seed=0):
    scores, gt, videos = np.asarray(scores, float), np.asarray(gt, int), np.asarray(videos)
    out = {"n": {"lying": int((gt == GT_LYING).sum()), "normal": int((gt == GT_NORMAL).sum()),
                 "transitional": int((gt == GT_TRANS).sum()),
                 "video_lying": len(set(videos[gt == GT_LYING]))},
           "thresholds": {}, "ap": average_precision(scores[gt == GT_LYING], scores[gt == GT_NORMAL])}
    by_vid = defaultdict(list)
    for i, v in enumerate(videos):
        by_vid[v].append(i)
    vids = sorted(by_vid)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(len(vids), size=len(vids), replace=True)
        idx = np.concatenate([by_vid[vids[k]] for k in pick])
        boots.append(idx)
    ap_b = [average_precision(scores[i][gt[i] == GT_LYING], scores[i][gt[i] == GT_NORMAL]) for i in boots]
    out["ap_ci"] = [float(np.nanpercentile(ap_b, 2.5)), float(np.nanpercentile(ap_b, 97.5))]
    for t in THRESHOLDS:
        m = metrics(scores, gt, t)
        rb = [metrics(scores[i], gt[i], t) for i in boots]
        for key in ("recall", "precision"):
            vals = [r[key] for r in rb if not np.isnan(r[key])]
            m[f"{key}_ci"] = [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [float("nan")] * 2
        out["thresholds"][f"{t:.2f}"] = m
    return out


def predict_scores(weights, images, imgsz, device, batch):
    from ultralytics import YOLO  # import di sini: skrip bisa diuji tanpa ultralytics
    model = YOLO(weights)
    ids = [k for k, v in model.names.items() if v == LYING_NAME]
    if len(ids) != 1:
        sys.exit(f"[FATAL] kelas '{LYING_NAME}' tidak ditemukan pada model: {model.names}")
    lid, scores = ids[0], []
    for i in range(0, len(images), batch):
        for r in model.predict([str(p) for p in images[i:i + batch]], conf=0.05, imgsz=imgsz,
                               device=device, verbose=False):
            b = r.boxes
            if b is None or len(b) == 0:
                scores.append(0.0)
                continue
            cls, conf = b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy()
            scores.append(float(conf[cls == lid].max()) if (cls == lid).any() else 0.0)
    return np.array(scores)


def print_report(res, title):
    n = res["n"]
    print(f"\n=== {title} ===")
    print(f"test: lying={n['lying']} (dari {n['video_lying']} video/skenario), normal={n['normal']}, transitional={n['transitional']}")
    print(f"AP lying-vs-normal = {res['ap']:.3f}  (95% CI per-video: {res['ap_ci'][0]:.3f} - {res['ap_ci'][1]:.3f})")
    print(f"{'conf':>5} {'recall':>8} {'[CI]':>15} {'precision':>10} {'[CI]':>15} {'FP normal':>10} {'FPR':>7} {'trans->lying':>13}")
    for t, m in res["thresholds"].items():
        print(f"{t:>5} {m['recall']:>8.3f} [{m['recall_ci'][0]:.2f},{m['recall_ci'][1]:.2f}] "
              f"{m['precision']:>10.3f} [{m['precision_ci'][0]:.2f},{m['precision_ci'][1]:.2f}] "
              f"{m['fp_normal']:>10d} {m['fpr_normal']:>7.3f} {m['trans_as_lying']:>13.3f}")


def main(argv=None):
    ap = argparse.ArgumentParser(description="Evaluasi lying vs bukan-lying (2 kelas / 3 kelas)")
    ap.add_argument("--weights", required=True, type=Path)
    ap.add_argument("--gt-dir", required=True, type=Path, help="Folder dataset 3 kelas (punya images/test, labels/test)")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--device", default="0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--boot", type=int, default=500)
    ap.add_argument("--out", type=Path, default=Path("eval_lying.json"))
    a = ap.parse_args(argv)

    img_dir, lbl_dir = a.gt_dir / "images" / "test", a.gt_dir / "labels" / "test"
    images = sorted(img_dir.glob("*.jpg"))
    if not images:
        sys.exit(f"[FATAL] tidak ada gambar di {img_dir}")
    gt, videos = [], []
    for p in images:
        lf = lbl_dir / f"{p.stem}.txt"
        if not lf.is_file():
            sys.exit(f"[FATAL] label hilang: {lf}")
        gt.append(int(lf.read_text().split()[0]))
        videos.append(video_of(p.stem))
    if set(gt) - {GT_NORMAL, GT_TRANS, GT_LYING} or GT_TRANS not in gt:
        sys.exit("[FATAL] --gt-dir harus dataset 3 kelas (kelas 0/1/2 semuanya ada)")
    scores = predict_scores(a.weights, images, a.imgsz, a.device, a.batch)
    res = evaluate(scores, gt, videos, n_boot=a.boot)
    res["weights"] = str(a.weights)
    a.out.write_text(json.dumps(res, indent=2))
    print_report(res, str(a.weights))
    print(f"\nTersimpan: {a.out}")


if __name__ == "__main__":
    main()
'''
Path("eval_lying.py").write_text(EVAL_LYING_CODE, encoding="utf-8")
sys.path.insert(0, str(Path(".").resolve()))
import eval_lying
importlib.reload(eval_lying)
print("eval_lying siap.")

In [ ]:
# ---- 6. Y1: training + evaluasi ----------------------------------------
all_rows, meta, RUN_DIRS = [], [], []
if RUN_Y1:
    for name, variant, wts, seed, *ov in RUNS:
        recipe = {**RECIPE, **(ov[0] if ov else {})}
        print("\n" + "=" * 70 + f"\nRUN {name} | {wts} | {variant} | seed {seed}\n" + "=" * 70)
        yaml = build_variant(variant)
        model = YOLO(wts)
        t0 = time.time()
        res = model.train(data=str(yaml), seed=seed, project=str(WORK / "runs"), name=name, exist_ok=True,
                          save=True, plots=True, **recipe)
        # pengaman: label yang dibaca Ultralytics harus label VARIAN di /kaggle/working, bukan label asli di /kaggle/input
        try:
            lf = str(model.trainer.train_loader.dataset.label_files[0])
            assert str(DS) in lf, f"label dibaca dari {lf}, bukan dari {DS}"
            print("label latih dibaca dari:", lf)
        except AttributeError:
            print("(peringatan: tidak bisa memeriksa asal label otomatis)")
        mins = (time.time() - t0) / 60
        RUN_DIRS.append(Path(res.save_dir))
        best = Path(res.save_dir) / "weights" / "best.pt"
        # metrik bawaan Ultralytics di test (varian terkait)
        m = YOLO(str(best)).val(data=str(yaml), split="test", imgsz=recipe["imgsz"], verbose=False)
        df, ms = eval_diag(best, variant, imgsz=recipe["imgsz"])
        ev_json = OUTD / f"{name}_eval_lying.json"
        eval_lying.main(["--weights", str(best), "--gt-dir", str(ROOT), "--imgsz", str(recipe["imgsz"]),
                         "--device", "0" if torch.cuda.is_available() else "cpu", "--batch", "16",
                         "--boot", "20" if SMOKE else "500", "--out", str(ev_json)])
        ev = json.loads(ev_json.read_text())
        t25, t50 = ev["thresholds"]["0.25"], ev["thresholds"]["0.50"]
        df.to_csv(OUTD / f"{name}_pred_test.csv", index=False)
        summ = summarize(df, name)
        all_rows.append(summ)
        meta.append(dict(run=name, variant=variant, weights=wts, seed=seed, minutes=round(mins, 1),
                         ms_per_image_eval=round(ms, 1), mAP50=round(float(m.box.map50), 4),
                         mAP50_95=round(float(m.box.map), 4), P=round(float(m.box.mp), 4), R=round(float(m.box.mr), 4),
                         AP_lying=round(ev["ap"], 3), AP_ci=str([round(x, 3) for x in ev["ap_ci"]]),
                         rec25=round(t25["recall"], 3), prec25=round(t25["precision"], 3), fp_normal25=t25["fp_normal"],
                         rec50=round(t50["recall"], 3), prec50=round(t50["precision"], 3), fp_normal50=t50["fp_normal"]))
        print(pd.DataFrame(meta[-1:]).T)
        print(summ[["kelompok", "lying_n", "top_lying_R", "top_lying_P", "any_lying_R", "any_lying_P", "lying_tak_terdeteksi"]].to_string(index=False))
        torch.cuda.empty_cache()
    if all_rows:
        pd.concat(all_rows).to_csv(OUTD / "y1_ringkasan_lying.csv", index=False)
        pd.DataFrame(meta).to_csv(OUTD / "y1_meta.csv", index=False)

In [ ]:
# ---- 7. Y0: YOLO26 tanpa training (pose / detektor orang) pada frame lying ----
lying_imgs = [ROOT / "images" / "test" / (l.stem + ".jpg") for l in sorted((ROOT / "labels" / "test").glob("*.txt"))
              if l.read_text().split()[0] == "2"]
print("frame lying di test:", len(lying_imgs))

def y0(mode, weights_list, sizes, conf=0.25, batch=8 if ON_KAGGLE else 4):
    out = []
    for w in weights_list:
        for s in sizes:
            try:
                model = YOLO(w)
                kw = dict(imgsz=s, conf=conf, verbose=False)
                if mode == "person":
                    kw["classes"] = [0]
                model([str(lying_imgs[0])], **kw)
                det = joint = low = 0
                percam = defaultdict(lambda: [0, 0])
                for i in range(0, len(lying_imgs), batch):
                    ch = lying_imgs[i:i + batch]
                    for p, r in zip(ch, model([str(x) for x in ch], **kw)):
                        cam = cam_of(p.name)
                        percam[cam][0] += 1
                        if len(r.boxes) == 0:
                            continue
                        det += 1
                        percam[cam][1] += 1
                        if mode == "pose":
                            b = int(r.boxes.conf.argmax())
                            k = r.keypoints.data[b].cpu().numpy()
                            if (k[5][2] + k[6][2]) / 2 > 0.25 and (k[11][2] + k[12][2]) / 2 > 0.25:
                                joint += 1
                                sx, sy = (k[5][0] + k[6][0]) / 2, (k[5][1] + k[6][1]) / 2
                                hx, hy = (k[11][0] + k[12][0]) / 2, (k[11][1] + k[12][1]) / 2
                                if math.degrees(math.atan2(abs(sy - hy), abs(sx - hx))) < 40:
                                    low += 1
                torch.cuda.synchronize()
                t0 = time.time()
                for p in lying_imgs[:40]:
                    model(str(p), **kw)
                torch.cuda.synchronize()
                ms = (time.time() - t0) / 40 * 1000
                n = len(lying_imgs)
                out.append(dict(mode=mode, weights=w, imgsz=s, terdeteksi=det, sendi_terbaca=joint, sudut_lt40=low, n=n,
                                pct_terdeteksi=round(det / n, 3), ms_per_gambar=round(ms, 1),
                                per_kamera={c: f"{v[1]}/{v[0]}" for c, v in sorted(percam.items())}))
                print(f"{mode:6s} {w:16s} {s:4d} | terdeteksi {det}/{n} ({det/n:.0%}) | sendi {joint} | sudut<40 {low} | {ms:.0f} ms")
            except Exception as e:
                print(f"{w} {s} GAGAL: {str(e)[:100]}")
            torch.cuda.empty_cache()
    return pd.DataFrame(out)

if RUN_Y0_POSE:
    y0p = y0("pose", Y0_WEIGHTS_POSE, Y0_IMGSZ if not SMOKE else [640])
    y0p.to_csv(OUTD / "y0_pose.csv", index=False)
if RUN_Y0_PERSON:
    y0q = y0("person", Y0_WEIGHTS_PERSON, Y0_IMGSZ if not SMOKE else [640])
    y0q.to_csv(OUTD / "y0_person.csv", index=False)

In [ ]:
# ---- 8. Simpan semua keluaran dalam satu zip -----------------------------
info = dict(ultralytics=ultralytics.__version__, torch=torch.__version__,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            dataset_fingerprint=got, smoke=SMOKE, recipe=RECIPE, runs=[list(r) for r in RUNS])
(OUTD / "info.json").write_text(json.dumps(info, indent=1))

zp = WORK / "y26_outputs.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUTD.glob("*"):
        z.write(f, f"out/{f.name}")
    for run_dir in RUN_DIRS:
        for rel in ("weights/best.pt", "results.csv", "results.png", "confusion_matrix.png",
                    "confusion_matrix_normalized.png", "args.yaml"):
            f = run_dir / rel
            if f.exists():
                z.write(f, f"runs/{run_dir.name}/{rel}")
print("Selesai. Unduh:", zp, f"({zp.stat().st_size/1e6:.1f} MB)")
if SMOKE:
    print("\nINI HANYA UJI CEPAT (SMOKE). Ubah SMOKE = False untuk run sebenarnya.")